Actual congruence closure

Thinnings +

Proof pruning



Hmm. So olveras style pruning seems like a big deal.
Makes sense that the trace is too much. Irrelvant rewrites infect.

path = path.



But how do I then think about when and where this happens




let id := f x y   
there is always a defining event. so we can associated transitively any id with some term

e1 -> e2 -> e3 -> e4


so cong and rewrite edges. No memo and uf path compress edges?


dump = default.

fun x -> e1 x = fun x -> e2. f = g
vs 
fun x -> e1 x = e2 x         forall x, f x = g x. Pointwise


proof producing thinning union find 

proof producing thinning egraph without emacthing / rewrites.

Give them as seperate data? That might let us do more.
Rel l1 l2 e1 e2 := l1 e1 = l2 e3

Give it as rewuiring the existence factoring
Eq l1 l2 e1 e2 :=  l1 e1 = l2 e3 /\ exists e3 f g, e1 = f e3 /\ e2 = g e3

FactorEq e1 e2 :=  exists e3 f g, e1 = f e3 /\ e2 = g e3

fun x y => e y  ==  fun x => e   is eta expansion

fun x y => e1 x = fun x y => e2 y


f ~ g  :=   forall x, f x = g x
homotopies as better?

Generalized congreunce closure / generalized rewriting?


In [ ]:
from dataclasses import dataclass, field
type Reason = tuple[object, bool] # reason and reversal
type FatId = tuple[Reason, int]
# type FatId = tuple[int, int] # src, dst

@dataclass
class UF:
    parents : list[FatId]
    def makeset(self) -> FatId: # hmm interesting. Doesn't work because what fatness. Actual path?
        self.parents.append(((None, False), len(self.parents)))
        return len(self.parents) - 1

@dataclass
class UF:
    parents : list[int] = field(default_factory=list)
    reasons : list[Reason] = field(default_factory=list)
    def makeset(self) -> int:
        self.parents.append(len(self.parents))
        self.reasons.append((None, False)) # refl
        return len(self.parents) - 1
    def find(self, x: int) -> int:
        while self.parents[x] != x:
            x = self.parents[x]
        return x # also return count for omre effiicnet rerooting choice?
    def find_chain(self, x: int) -> list[int]:
        chain = [x]
        while self.parents[x] != x:
            x = self.parents[x]
            chain.append(x)
        return chain
    def reroot(self, x : int):
        # hmm. so there is a version that reverses as you go up.
        chain = self.find_chain(x)
        N = len(chain)
        for i in range(N-1): # reverse direction 
            x,y = chain[N-i-2], chain[N-i-1]
            self.parents[y] = x 
            reason,rev = self.reasons[x]
            self.reasons[y] = (reason, not rev)
    def union(self, x : int, y : int, reason) -> bool:
        x1,y1 = self.find(x), self.find(y) # cheap find first
        if x1 != y1:
            self.reroot(x)
            self.parents[x] = y
            self.reasons[x] = (reason, False)
    def explain(self, x : int, y : int) -> tuple[list[Reason], list[Reason]]:
        assert self.find(x) == self.find(y)
        chain_x = self.find_chain(x)
        chain_y = self.find_chain(y)
        reasons_x = [self.reasons[node] for node in chain_x[:-1]] # [:-1] to ignore refl at top? Eh.
        reasons_y = [self.reasons[node] for node in chain_y[:-1]]
        reasons = reasons_x + [(r, not rev) for r, rev in reversed(reasons_y)]
        # could also pop True, False pairs
        return reasons
        #return reasons_x, reasons_y
        #reason = [self.reasons[node] for node in chain_x[:-1]]
        #for r in reversedchain_y[:-1]:
        #    reason.append((r, not rev))
uf = UF()

x,y,z = uf.makeset(), uf.makeset(), uf.makeset()
uf.union(x,y, "x->y")
uf.union(x, z, "x->z")
uf.explain(x,y)